# 00 - Preparação

In [8]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


**Parte 1 - Inicialização e Marcadores Básicos**

In [ ]:
centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

mapa = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

mapa

In [ ]:
for _, imovel in df_mapa.head(5).iterrows():
    folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=f"Tipo: {imovel['tipo']}<br>Valor: R$ {imovel['valor_venda']:,.2f}"
    ).add_to(mapa)

mapa

**Parte 2 - Customização Visual com Marcadores Circulares**

In [ ]:
centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

novoMapa = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, imovel in df_mapa.iterrows():

    if imovel['cidade'] == 'Nova Iguaçu':
        cor = 'blue'
    else:
        cor = 'orange'

    folium.CircleMarker(
        location=[imovel['latitude'], imovel['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        tooltip='Clique para detalhes'
    ).add_to(novoMapa)

novoMapa

**Parte 3 - Agrupamento Inteligente**

In [ ]:
centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

mapa3 = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

cluster = MarkerCluster().add_to(mapa3)

for _, imovel in df_mapa.iterrows():

    if imovel['tipo'] == 'Casa':
        cor = 'green'
    elif imovel['tipo'] == 'Apartamento':
        cor = 'blue'
    else:
        cor = 'gray'

    folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        icon=folium.Icon(color=cor)
    ).add_to(cluster)

mapa3.save('mapa_imoveis_baixada.html')

mapa3